<a href="https://colab.research.google.com/github/obeabi/ProjectPortfolio/blob/master/EarningsLab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [76]:
!pip install --upgrade yfinance
!pip install scipy==1.16.2

In [78]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
from math import sqrt
print("Libraries installed successfully!")

Libraries installed successfully!


## Build earnings lab class

In [92]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime

def earnings_edge_engine(ticker_symbol, lookback=12):
    """
      tickers: str or list of ticker symbols
      Computes full event metrics: gap, intraday, 2-day reaction, top moves, volatility bias
      lookback: number of past earnings to analyze
      Returns: pd.DataFrame with one row per ticker
    """

    ticker = yf.Ticker(ticker_symbol)

    # -------------------------
    # 1️⃣ Pull Earnings Dates
    # -------------------------
    earnings = ticker.get_earnings_dates(limit=lookback)

    if earnings is None or earnings.empty:
        print("No earnings data found.")
        return

    earnings = earnings.reset_index()
    earnings['Earnings Date'] = earnings['Earnings Date'].dt.tz_localize(None)
    today = pd.Timestamp.today()

    # Separate past and future earnings
    past_earnings = earnings[earnings['Earnings Date'] < today]
    future_earnings = earnings[earnings['Earnings Date'] >= today]

    if future_earnings.empty:
        print("No upcoming earnings found.")
        return

    next_earnings_date = future_earnings.iloc[0]['Earnings Date'].date()

    # -------------------------
    # 2️⃣ Historical Move Calculation
    # -------------------------
    price_data = ticker.history(period="5y")
    price_data.index = price_data.index.date

    results = []

    for edate in past_earnings['Earnings Date'].dt.date:

        if edate not in price_data.index:
            continue
        trading_days = sorted(price_data.index)

        if edate not in trading_days:
            continue

        edate_index = trading_days.index(edate)

        # Need at least 1 prior date and 1 next day
        if edate_index == 0 or edate_index == len(trading_days) - 1:
            continue

        prior_days = [d for d in price_data.index if d < edate]
        if not prior_days:
            continue

        #prior_day = max(prior_days)

        prior_day = trading_days[edate_index - 1]
        next_day = trading_days[edate_index + 1]
        open_price = price_data.loc[edate]['Open']
        high_price = price_data.loc[edate]['High']
        low_price = price_data.loc[edate]['Low']

        prior_close = price_data.loc[prior_day]['Close']
        earnings_close = price_data.loc[edate]['Close']
        next_close = price_data.loc[next_day]['Close']


         # Institutional metrics
        #close_to_close_pct = (next_close - prior_close) / prior_close * 100
        gap_pct = (open_price - prior_close)/prior_close * 100
        intraday_move_pct = (high_price - low_price)/prior_close * 100
        day1_close_to_prior_close_pct = (earnings_close - prior_close)/prior_close * 100
        two_day_reaction_pct = (next_close - prior_close)/prior_close * 100




        #earnings_close = price_data.loc[edate]['Close']
        #next_close = price_data.loc[next_day]['Close']
        #results.append(close_to_close_pct)
        results.append({
        "Earnings Date": edate,
        "Prior Day": prior_day,
        "Reaction Day": next_day,
        "Gap %": gap_pct,
        "Intraday %": intraday_move_pct,
        "Day1 Close %": day1_close_to_prior_close_pct,
        "Two Day Reaction %": two_day_reaction_pct,
        "Direction": "Up" if two_day_reaction_pct>0 else "Down",
        "Abs Two Day Reaction %": abs(two_day_reaction_pct)
        #"Prior Day": prior_day,
        #"Reaction Day": next_day,
        #"Two Day Reaction %": two_day_reaction_pct
        })


    if len(results) == 0:
        print("No valid historical earnings moves.")
        return

    #moves = pd.Series(results).abs()
    df_moves = pd.DataFrame(results)
    df_moves['Abs Move %'] = df_moves['Two Day Reaction %'].abs()


    #hist_avg = df_moves["Abs Move %"].mean()
    #hist_median = df_moves["Abs Move %"].median()
    #hist_std = df_moves["Abs Move %"].std()
    #max_row = df_moves.loc[df_moves["Abs Move %"].idxmax()]
    #hist_max = max_row["Abs Move %"]
    #max_move_date = max_row["Earnings Date"]
    hist_avg = df_moves["Abs Two Day Reaction %"].mean()
    hist_median = df_moves["Abs Two Day Reaction %"].median()
    hist_std = df_moves["Abs Two Day Reaction %"].std()
    max_row = df_moves.loc[df_moves["Abs Two Day Reaction %"].idxmax()]
    hist_max = max_row["Abs Two Day Reaction %"]
    max_move_date = max_row["Earnings Date"]
    max_move_direction = max_row["Direction"]
    top3 = df_moves.nlargest(3, 'Abs Two Day Reaction %')[['Earnings Date','Two Day Reaction %','Direction']]

    # -------------------------
    # 3️⃣ Select Correct Expiration (AFTER earnings)
    # -------------------------
    expirations = ticker.options
    if not expirations:
        print("No options data available.")
        return

    exp_dates = [pd.to_datetime(e).date() for e in expirations]

    valid_exps = [e for e in exp_dates if e > next_earnings_date]

    if not valid_exps:
        print("No expiration found after earnings.")
        return

    target_exp = min(valid_exps)

    option_chain = ticker.option_chain(str(target_exp))
    calls = option_chain.calls
    puts = option_chain.puts

    current_price = ticker.history(period="1d")["Close"].iloc[-1]

    # Find ATM strike
    calls["distance"] = abs(calls["strike"] - current_price)
    puts["distance"] = abs(puts["strike"] - current_price)

    atm_call = calls.loc[calls["distance"].idxmin()]
    atm_put = puts.loc[puts["distance"].idxmin()]

    straddle_price = atm_call["lastPrice"] + atm_put["lastPrice"]
    implied_move_pct = (straddle_price / current_price) * 100

    # -------------------------
    # 4️⃣ Volatility Bias Logic
    # -------------------------
    ratio = implied_move_pct / hist_avg

    if ratio > 1.25:
        bias = "SELL VOLATILITY (Premium Overpriced)"
        suggested_structure = "Short Straddle / Short Iron Condor"
    elif ratio < 0.80:
        bias = "BUY VOLATILITY (Premium Underpriced)"
        suggested_structure = "Long Straddle / Debit Spread"
    else:
        bias = "NEUTRAL / FAIR"
        suggested_structure = "Directional or Calendar Spread"

    bias_score = round((ratio - 1) * 100, 2)

    # -------------------------
    # 5️⃣ Output
    # -------------------------
    print(f"\n===== Earnings Edge Analysis: {ticker_symbol} =====")
    print(f"Next Earnings Date: {next_earnings_date}")
    print(f"Option Expiration Used: {target_exp}")
    print(f"\nCurrent Price: {round(current_price,2)}")
    print(f"Implied Move % (ATM Straddle): {round(implied_move_pct,2)}%")

    #print("\n--- Historical Earnings Moves (Abs Close-to-Close %) ---")
    #print(f"Average Move: {round(hist_avg,2)}%")
    #print(f"Median Move: {round(hist_median,2)}%")
    #print(f"Max Move: {round(hist_max,2)}%")
    #print(f"Std Dev: {round(hist_std,2)}%")
    print("\n--- Historical Earnings Moves ---")
    print(f"Average Move: {round(hist_avg,2)}%")
    print(f"Median Move: {round(hist_median,2)}%")
    print(f"Max Move: {round(hist_max,2)}%")
    print(f"Max Move Date: {max_move_date}")
    print(f"Std Dev: {round(hist_std,2)}%")

    print("\n--- Volatility Assessment ---")
    print(f"Implied / Historical Ratio: {round(ratio,2)}")
    print(f"Volatility Bias Score: {bias_score}")
    print(f"Suggested Action: {bias}")
    summary_df = pd.DataFrame([{
      "Ticker": ticker_symbol,
      "Next Earnings": next_earnings_date,
      "Expiration Used": target_exp,
      "Current Price": round(current_price, 2),
      "Implied Move %": round(implied_move_pct, 2),
      "Historical Avg %": round(hist_avg, 2),
      "Historical Median %": round(hist_median, 2),
      "Historical Max %": round(hist_max, 2),
      "Max Move Date": max_move_date,
      "Std Dev %": round(hist_std, 2),
      "Implied/Hist Ratio": round(ratio, 2),
      "Volatility Bias Score": bias_score,
     "Suggested Action": bias,
      "Suggested Structure": suggested_structure,
      "Top3 Moves": top3.to_dict(orient='records')
      }])

    return summary_df



In [93]:

def earnings_edge_batch(tickers, lookback=12):
    """
    tickers: str or list of ticker symbols
    Returns: pd.DataFrame with one row per ticker
    """
    if isinstance(tickers, str):
        tickers = [tickers]

    all_results = []

    for ticker_symbol in tickers:
        try:
            df = earnings_edge_engine(ticker_symbol, lookback)
            if df is not None:
                all_results.append(df)
        except Exception as e:
            print(f"Error processing {ticker_symbol}: {e}")
            continue

    if not all_results:
        return pd.DataFrame()  # empty DataFrame

    return pd.concat(all_results, ignore_index=True)

In [94]:
my_list = ["NVDA", "HD", "O", "SNOW", "CRM", "WBD", "DELL", "BIDU", "VST", "INT","BRK.B"]
df_results = earnings_edge_batch(my_list)
df_results
#df = earnings_edge_engine(my_list)
#df


===== Earnings Edge Analysis: NVDA =====
Next Earnings Date: 2026-02-25
Option Expiration Used: 2026-02-27

Current Price: 191.55
Implied Move % (ATM Straddle): 6.25%

--- Historical Earnings Moves ---
Average Move: 6.34%
Median Move: 4.87%
Max Move: 23.76%
Max Move Date: 2023-05-24
Std Dev: 5.95%

--- Volatility Assessment ---
Implied / Historical Ratio: 0.99
Volatility Bias Score: -1.31
Suggested Action: NEUTRAL / FAIR

===== Earnings Edge Analysis: HD =====
Next Earnings Date: 2026-02-24
Option Expiration Used: 2026-02-27

Current Price: 376.99
Implied Move % (ATM Straddle): 4.24%

--- Historical Earnings Moves ---
Average Move: 3.57%
Median Move: 2.6%
Max Move: 11.08%
Max Move Date: 2022-02-22
Std Dev: 2.79%

--- Volatility Assessment ---
Implied / Historical Ratio: 1.19
Volatility Bias Score: 18.7
Suggested Action: NEUTRAL / FAIR

===== Earnings Edge Analysis: O =====
Next Earnings Date: 2026-02-24
Option Expiration Used: 2026-03-20

Current Price: 66.68
Implied Move % (ATM Strad

ERROR:yfinance:INT: No earnings dates found, symbol may be delisted



===== Earnings Edge Analysis: VST =====
Next Earnings Date: 2026-02-26
Option Expiration Used: 2026-02-27

Current Price: 167.8
Implied Move % (ATM Straddle): 8.02%

--- Historical Earnings Moves ---
Average Move: 4.7%
Median Move: 3.2%
Max Move: 23.51%
Max Move Date: 2021-02-26
Std Dev: 4.8%

--- Volatility Assessment ---
Implied / Historical Ratio: 1.71
Volatility Bias Score: 70.63
Suggested Action: SELL VOLATILITY (Premium Overpriced)
No earnings data found.


ERROR:yfinance:BRK.B: No earnings dates found, symbol may be delisted


No earnings data found.


,Ticker,Next Earnings,Expiration Used,Current Price,Implied Move %,Historical Avg %,Historical Median %,Historical Max %,Max Move Date,Std Dev %,Implied/Hist Ratio,Volatility Bias Score,Suggested Action,Suggested Structure,Top3 Moves
0,NVDA,2026-02-25,2026-02-27,191.55,6.25,6.34,4.87,23.76,2023-05-24,5.95,0.99,-1.31,NEUTRAL / FAIR,Directional or Calendar Spread,"[{'Earnings Date': 2023-05-24, 'Two Day Reacti..."
1,HD,2026-02-24,2026-02-27,376.99,4.24,3.57,2.60,11.08,2022-02-22,2.79,1.19,18.70,NEUTRAL / FAIR,Directional or Calendar Spread,"[{'Earnings Date': 2022-02-22, 'Two Day Reacti..."
2,O,2026-02-24,2026-03-20,66.68,4.30,1.01,0.84,3.19,2023-08-02,0.96,4.27,326.63,SELL VOLATILITY (Premium Overpriced),Short Straddle / Short Iron Condor,"[{'Earnings Date': 2023-08-02, 'Two Day Reacti..."
3,SNOW,2026-02-25,2026-02-27,157.60,14.31,12.34,10.35,31.56,2024-11-20,8.07,1.16,15.91,NEUTRAL / FAIR,Directional or Calendar Spread,"[{'Earnings Date': 2024-11-20, 'Two Day Reacti..."
4,CRM,2026-02-25,2026-02-27,178.16,9.80,6.43,3.69,19.20,2024-05-29,5.28,1.52,52.36,SELL VOLATILITY (Premium Overpriced),Short Straddle / Short Iron Condor,"[{'Earnings Date': 2024-05-29, 'Two Day Reacti..."
5,WBD,2026-02-26,2026-02-27,28.92,4.94,8.70,9.14,19.79,2022-08-05,5.81,0.57,-43.14,BUY VOLATILITY (Premium Underpriced),Long Straddle / Debit Spread,"[{'Earnings Date': 2022-08-05, 'Two Day Reacti..."
6,DELL,2026-02-26,2026-02-27,119.14,10.77,9.13,5.25,33.61,2024-02-29,8.59,1.18,17.90,NEUTRAL / FAIR,Directional or Calendar Spread,"[{'Earnings Date': 2024-02-29, 'Two Day Reacti..."
7,BIDU,2026-02-26,2026-02-27,133.94,7.97,5.41,5.33,16.57,2022-05-26,4.26,1.47,47.12,SELL VOLATILITY (Premium Overpriced),Short Straddle / Short Iron Condor,"[{'Earnings Date': 2022-05-26, 'Two Day Reacti..."
8,VST,2026-02-26,2026-02-27,167.80,8.02,4.70,3.20,23.51,2021-02-26,4.80,1.71,70.63,SELL VOLATILITY (Premium Overpriced),Short Straddle / Short Iron Condor,"[{'Earnings Date': 2021-02-26, 'Two Day Reacti..."
